In [1]:
from figgie_gym.envs.common import SUITS
import numpy as np
from sklearn.metrics import log_loss

In [2]:
import pandas as pd

full_df = pd.read_parquet("/Users/yawerijaz/figgie_gym/data/multi_debug/game_runs")
full_df['coalesced_type'] = full_df['agent_info.agent_params.model_class'].fillna(full_df['agent_info.agent_params.agent_type'])


In [3]:
df = full_df
belief_cols = [f"belief.{s}" for s in SUITS]
is_agent_correct = (df[belief_cols].idxmax(axis=1).str.removeprefix("belief.") == df["game_info.goal_suit"])
df = df.assign(
    is_agent_correct=is_agent_correct,
)
df['coalesced_type'].value_counts()

coalesced_type
NoiseAgent                  617542
SuitAgentEquivClassifier    312010
EquivariantClassifier       310165
FCTensorDictClassifier      307869
NaiveClassifier             299464
CardCounterAgent            202950
Name: count, dtype: int64

In [4]:
df.pivot_table(
    # index="current_observations.time",
    index='game_info.agent_type_count.NoiseAgent',
    columns='coalesced_type',
    values=["is_agent_correct"],
)

is_agent_correct                        \
coalesced_type                        CardCounterAgent EquivariantClassifier   
game_info.agent_type_count.NoiseAgent                                          
1.0                                           0.544454              0.708133   
2.0                                           0.523548              0.615587   
3.0                                           0.545568              0.621972   
4.0                                           0.601428              0.666979   
5.0                                                NaN                   NaN   

                                                                              \
coalesced_type                        FCTensorDictClassifier NaiveClassifier   
game_info.agent_type_count.NoiseAgent                                          
1.0                                                 0.547528        0.251230   
2.0                                                 0.548184        0.253708   
3.0                                                 0.580953        0.237024   
4.0                                                 0.590036        0.208333   
5.0                                                      NaN             NaN   

                                                                           
coalesced_type                        NoiseAgent SuitAgentEquivClassifier  
game_info.agent_type_count.NoiseAgent                                      
1.0                                     0.254284                 0.615792  
2.0                                     0.266161                 0.575628  
3.0                                     0.242313                 0.611158  
4.0                                     0.237410                 0.650915  
5.0                                     0.352941                      NaN

In [5]:
alias = {    
    "current_observations.time": "time",
    'game_info.agent_type_count.NoiseAgent': "num_noise_agents",
    'coalesced_type': "agent_type",
    "is_agent_correct": "accuracy",
}


def cal_log_loss(group_df: pd.DataFrame) -> float:
    belief = group_df[belief_cols] / group_df[belief_cols].sum(axis=1).values[:, np.newaxis]
    return log_loss(group_df["game_info.goal_suit_code"], belief)

g = df.assign(
    is_agent_correct=is_agent_correct,
).groupby([
    "current_observations.time",
    'game_info.agent_type_count.NoiseAgent',
    'coalesced_type',
])
accuracy_summary = g["is_agent_correct"].mean().reset_index().rename(columns=alias)
log_loss_summary = pd.DataFrame([{"time": time, "num_noise_agents": num_noise_agents, "agent_type": agent_type, "log_loss": cal_log_loss(group_df)} for (time, num_noise_agents, agent_type), group_df in g])


In [6]:
import plotly.express as px
px.scatter(accuracy_summary, x="time", y="accuracy", color="num_noise_agents", facet_col="agent_type", title="Accuracy").show()
px.scatter(log_loss_summary, x="time", y="log_loss", color="num_noise_agents", facet_col="agent_type", title="Log Loss").show()